In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

In [2]:
input_shape = (28, 28, 1)
num_classes = 10

In [3]:
(x_train, y_train), (x_test, y_test) = datasets.fashion_mnist.load_data()

In [5]:
def myModel(input_shape, num_classes):    
    inputs = layers.Input(shape=input_shape)
    shortcut = inputs # 추가 숏컷으로 쓸거
    net = layers.Conv2D(32, 3, 1, padding='same', activation='relu')(inputs)
    net = layers.Conv2D(64, 3, 1, padding='same', activation='relu')(net)
    
    shortcut = layers.Conv2D(64, 1, 1, padding='same', activation='relu')(shortcut)

    net = shortcut + net
    
    net = layers.Flatten()(net)
    net = layers.Dense(512, activation='relu')(net)
    net = layers.Dense(num_classes, activation='softmax')(net)
    
    model = tf.keras.Model(inputs=inputs, outputs=net, name = 'TestModel') # 요거 이름 공란 있으면 밑에서 오류남 

    return model

In [6]:
model = myModel(input_shape, num_classes)

In [7]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss = 'sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [8]:
x_train_4d = x_train[..., tf.newaxis] # 3D -> 4D
print(x_train_4d.shape, x_train.shape, x_train_4d.dtype)
x_train_4d = x_train_4d / 255.0 # scaling to [0,1]
print(np.max(x_train_4d), np.min(x_train_4d))

(60000, 28, 28, 1) (60000, 28, 28) uint8
1.0 0.0


In [9]:
x_test_4d = x_test[..., tf.newaxis] # 3D -> 4D
x_test_4d = x_test_4d / 255.0
print(x_test_4d.shape)

(10000, 28, 28, 1)


In [10]:
num_epochs = 10
bs = 32
hist = model.fit(x_train_4d, y_train, validation_data = (x_test_4d, y_test), batch_size=bs, shuffle=True, epochs=num_epochs)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 100s 53ms/step - accuracy: 0.8701 - loss: 0.3673 - val_accuracy: 0.8965 - val_loss: 0.2877
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 98s 52ms/step - accuracy: 0.9260 - loss: 0.1997 - val_accuracy: 0.9076 - val_loss: 0.2636
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 98s 53ms/step - accuracy: 0.9536 - loss: 0.1246 - val_accuracy: 0.9122 - val_loss: 0.2709
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 142s 53ms/step - accuracy: 0.9726 - loss: 0.0745 - val_accuracy: 0.9129 - val_loss: 0.3022
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - accuracy: 0.9851 - loss: 0.0416 - val_accuracy: 0.9142 - val_loss: 0.3776
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - accuracy: 0.9906 - loss: 0.0276 - val_accuracy: 0.9150 - val_loss: 0.4137
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - accuracy: 0.9929 - loss: 0.0201 - val_accuracy: 0.9158 - val_loss: 0.5183
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - accuracy: 0.9941

### ResNet blocks

In [26]:
def residual_block(x, filters, stride = 1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides = stride, padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, strides = 1, padding = 'same')(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides = stride, padding = 'same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut]) # x.shape == shortcut.shape 이어야 함
    x = layers.ReLU()(x)

    return x

In [27]:
def myModel(input_shape, filters, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)

    net = layers.Conv2D(filters, 3, 1, padding='same')(inputs)
    net = layers.BatchNormalization()(net)
    net = layers.ReLU()(net)

    net = residual_block(net, 64)
    net = residual_block(net, 64)
    net = residual_block(net, 128, stride = 2) # half size
    net = residual_block(net, 128)
    net = residual_block(net, 256, stride = 2) # half size
    net = residual_block(net, 256)

    net = layers.GlobalAveragePooling2D()(net)

    net = tf.keras.layers.Dense(num_classes, activation='softmax')(net)

    model = tf.keras.Model(inputs=inputs, outputs=net, name='test')

    return(model)

In [21]:
model.summary()

Model: "TestModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 28, 28, 1)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 28, 28, 32)        │             320 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 28, 28, 64)        │             128 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 28, 28, 64)        │          18,496 │ conv2d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 28, 28, 64)        │               0 │ conv2d_2[0][0],            │
│                               │                           │                 │ conv2d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 50176)             │               0 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 512)               │      25,690,624 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 10)                │           5,130 │ dense[0][0]                │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 77,144,096 (294.28 MB)

 Trainable params: 25,714,698 (98.09 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 51,429,398 (196.19 MB)

In [15]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 86s 1us/step


In [17]:
x_train_4d = x_train / 255.0 # scaling to [0,1]
x_test_4d = x_test / 255.0 # scaling to [0,1]

In [18]:
x_train_4d.shape, x_test_4d.shape

((50000, 32, 32, 3), (10000, 32, 32, 3))

In [28]:
input_shape = (32, 32, 3)
filters =32
model = myModel(input_shape, filters, num_classes)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss = 'sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [ ]:
num_epochs = 10
bs = 32
hist = model.fit(x_train_4d, y_train, validation_data = (x_test_4d, y_test), batch_size=bs, shuffle=True, epochs=num_epochs)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 142s 88ms/step - accuracy: 0.5465 - loss: 1.2641 - val_accuracy: 0.4579 - val_loss: 1.7225
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 153s 98ms/step - accuracy: 0.7271 - loss: 0.7775 - val_accuracy: 0.5943 - val_loss: 1.3617
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 157s 100ms/step - accuracy: 0.7949 - loss: 0.5959 - val_accuracy: 0.6461 - val_loss: 1.1525
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 156s 100ms/step - accuracy: 0.8381 - loss: 0.4697 - val_accuracy: 0.7558 - val_loss: 0.7412
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 158s 101ms/step - accuracy: 0.8678 - loss: 0.3825 - val_accuracy: 0.7813 - val_loss: 0.6848
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 156s 100ms/step - accuracy: 0.8980 - loss: 0.2990 - val_accuracy: 0.7708 - val_loss: 0.7261
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 160s 102ms/step - accuracy: 0.9209 - loss: 0.2252 - val_accuracy: 0.8001 - val_loss: 0.6569
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 156s 100ms/step - accu